# Double-transmission protocol

In this file, Alice will compute $E_u$ and $E_v$, and send both sums back to Bob. Bob will then make a guess based on which of $E - S/n$ is closer to 0 as to which one represents distribution U and inform Alice to select the bit associated with that transmission.

For example, if Bob believes $E_1$ (public name) to be the $E_u$ distribution, he will select 1 as his bit. He informs Alice of his choice, and if he was right then they agree. If not, Alice will receive back the distribution associated with V and therefore select 0.

- Notice that this protocol diverges from the previous one in that Bob no longer experimentally determines a threshold.

- Distribution U is an exponential distribution, whereas Distribution V is uniform.


In [1]:
import numpy as np
import pandas as pd
import csv, os, datetime as dt
import matplotlib.pyplot as plt

In [20]:
def run_protocol(n=200000, k=8, q=50, mu1=20000, mu2=25000,
                 S_start=10_000_000, S_end=100_000_000,
                 num_trials=2000, seed=None):
    
    """Run one full experiment and return a summary dict (one notebook run = one call).

    A fresh np.random.Generator is created from `seed`, so runs are isolated and
    reproducible. Pass seed=None for a different draw each time.
   
    n          --> number of total samples
    k          --> size of set U
    q          --> size of Bob's selection from U
    mu1        --> mean of distribution U
    mu2        --> mean of distribution V
    S_start    --> lower bound for Bob's private key
    S_end      --> upper bound for Bob's private key
    num_trials --> number of iterations the protocol is run for
    seed       --> set seed for reproducibility
    
    """
    rng = np.random.default_rng(seed)
    U = rng.exponential(scale=mu1, size=k)                     # public set, fixed for this run - If you want the same U for every
                                                               # trial, move U to empty cell above this function and ensure you
                                                               # edit the k value.

    successes = correct_U = total_U = correct_V = total_V = 0
    bob_tot = alice_tot = 0.0

    for _ in range(num_trials):
        bob_sum = rng.choice(U, size=q).sum()
        S = rng.integers(S_start, S_end + 1)
        bob_tx = bob_sum + S                                  # Bob -> Alice (S masks bob_sum)

        
        alice_sum_U = rng.exponential(scale=mu1, size=n - q).sum() + bob_tx
        alice_sum_V = np.random.randint(0, 2*mu2, size=n-q, dtype=int).sum() + bob_tx #note that Alice creates both sets
        
        E_1 = alice_sum_U/n - mu1 #Alice calculates what to send back to Bob
        E_2 = alice_sum_V/n - mu1 #Alice ---> Bob

        if abs(E_1 - S/n) <= abs(E_2 - S/n):
            bob_guess_U = E_1
            bob_guess_V = E_2
        else:
            bob_guess_U = E_2
            bob_guess_V = E_1

        bob_bit = 1 if rng.random() < 0.5 else 0 

        if bob_bit == 1: #alice receives bob_guess_U
            if bob_guess_U == E_1:
               alice_bit = 1
            else:
               alice_bit = 0
            
        else: #alice receives bob_guess_V
            if bob_guess_V == E_2:
                alice_bit = 0
            else:
                alice_bit = 1
        #Now count success

        successes += (bob_bit == alice_bit)
        if bob_guess_U == E_1:
            total_U += 1; correct_U += (bob_bit == alice_bit)
        else:
            total_V += 1; correct_V += (bob_bit == alice_bit)
        

    return {
        "success_rate":       successes / num_trials,
        "S_start":            S_start,
        "S_end":              S_end,
        "S_range":            f"{-S_start+S_end}",
        "q":                  q,
        "n":                  n,
        "mu1":                mu1,
        "mu2":                mu2,
        "num_trials":         num_trials,
        "acc_given_U":        correct_U / total_U if total_U else float("nan"), #accuracy when bob guesses E_1
        "acc_given_V":        correct_V / total_V if total_V else float("nan"), #Accuracy when bob guesses E_2
        "Set U":              U,
        "Mean of U":          sum(U)/len(U),
        "seed":               seed,
    }

In [21]:
def append_run(summary, log_path="double_transmission_run_log.csv"):
    """Append one summary dict as a row, writing the header only if the file is new."""
    record = {"timestamp": dt.datetime.now().isoformat(timespec="seconds"), **summary}
    write_header = not os.path.isfile(log_path)
    with open(log_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(record.keys()))
        if write_header:
            w.writeheader()
        w.writerow(record)
    return record

## Single run
Run the protocol once and append the summary row to `notebook_run_log.csv`.

In [19]:
summary = run_protocol(S_start=1_000_000, S_end=10_000_000, q=100, mu1=20_000, mu2=20_160, seed=None)     # defaults: n=200000, q=250, num_trials=2000
                                   # run_protocol(n=200000, q=50, k=8, mu1=20000, mu2=50000,
                                   # S_start=10_000_000, S_end=100_000_000,
                                   # num_trials=2000, seed=None):
append_run(summary)
summary

{'success_rate': 0.998,
 'S_start': 10000,
 'S_end': 1000000,
 'S_range': '990000',
 'q': 100,
 'n': 200000,
 'mu1': 20000,
 'mu2': 20160,
 'num_trials': 2000,
 'acc_given_U': 1.0,
 'acc_given_V': 0.0,
 'Set U': array([ 1155.89414832, 21481.37181352, 21200.30040043,  6203.19332779,
         4996.72262693,   327.47392816, 21855.45646974, 50108.01846935]),
 'Mean of U': np.float64(15916.053898029511),
 'seed': None}

# Eavesdropping
Eve is an eavesdropper who has access only to public keys and information shared publicly by the communicating parties Bob and Alice. I will add in an eavesdropper to the above protocol.

In [27]:
def run_protocol_eve(n=200000, k=8, q=50, mu1=20000, mu2=25000,
                 S_start=10_000_000, S_end=100_000_000,
                 num_trials=2000, seed=None):
    
    """Run one full experiment and return a summary dict (one notebook run = one call).

    A fresh np.random.Generator is created from `seed`, so runs are isolated and
    reproducible. Pass seed=None for a different draw each time.
   
    n          --> number of total samples
    k          --> size of set U
    q          --> size of Bob's selection from U
    mu1        --> mean of distribution U
    mu2        --> mean of distribution V
    S_start    --> lower bound for Bob's private key
    S_end      --> upper bound for Bob's private key
    num_trials --> number of iterations the protocol is run for
    seed       --> set seed for reproducibility
    
    """
    rng = np.random.default_rng(seed)
    U = rng.exponential(scale=mu1, size=k)                     # public set, fixed for this run - If you want the same U for every
                                                               # trial, move U to empty cell above this function and ensure you
                                                               # edit the k value.

    successes = correct_U = total_U = correct_V = total_V = 0
    bob_tot = alice_tot = 0.0
    eve_success = 0
    
    for _ in range(num_trials):
        bob_sum = rng.choice(U, size=q).sum()
        S = rng.integers(S_start, S_end + 1)
        bob_tx = bob_sum + S                                  # Bob -> Alice (S masks bob_sum)

        
        alice_sum_U = rng.exponential(scale=mu1, size=n - q).sum() + bob_tx
        alice_sum_V = np.random.randint(0, 2*mu2, size=n-q, dtype=int).sum() + bob_tx #note that Alice creates both sets
        
        E_1 = alice_sum_U/n - mu1 #Alice calculates what to send back to Bob
        E_2 = alice_sum_V/n - mu1 #Alice ---> Bob

        if abs(E_1 - S/n) <= abs(E_2 - S/n):
            bob_guess_U = E_1
            bob_guess_V = E_2
        else:
            bob_guess_U = E_2
            bob_guess_V = E_1

        bob_bit = 1 if rng.random() < 0.5 else 0 

        if bob_bit == 1: #alice receives bob_guess_U
            if bob_guess_U == E_1:
               alice_bit = 1
            else:
               alice_bit = 0
            
        else: #alice receives bob_guess_V
            if bob_guess_V == E_2:
                alice_bit = 0
            else:
                alice_bit = 1
        #Now count success

        successes += (bob_bit == alice_bit)
        if bob_guess_U == E_1:
            total_U += 1; correct_U += (bob_bit == alice_bit)
        else:
            total_V += 1; correct_V += (bob_bit == alice_bit)

        #Now enter Eve
        eve_guess_U = min(E_1, E_2)
        eve_guess_V = max(E_1, E_2)

        eve_success += (eve_guess_U == bob_guess_U)
            
 
    return {
        "success_rate":       successes / num_trials,
        "Eve success rate":   eve_success / num_trials,
        "S_start":            S_start,
        "S_end":              S_end,
        "S_range":            f"{-S_start+S_end}",
        "q":                  q,
        "n":                  n,
        "mu1":                mu1,
        "mu2":                mu2,
        "num_trials":         num_trials,
        "acc_given_U":        correct_U / total_U if total_U else float("nan"), #accuracy when bob guesses E_1
        "acc_given_V":        correct_V / total_V if total_V else float("nan"), #Accuracy when bob guesses E_2
        "Set U":              U,
        "Mean of U":          sum(U)/len(U),
        "seed":               seed,
    }

In [24]:
def append_run_eve(summary, log_path="EAVESDROPPER_double_transmission_run_log.csv"):
    """Append one summary dict as a row, writing the header only if the file is new."""
    record = {"timestamp": dt.datetime.now().isoformat(timespec="seconds"), **summary}
    write_header = not os.path.isfile(log_path)
    with open(log_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(record.keys()))
        if write_header:
            w.writeheader()
        w.writerow(record)
    return record

In [30]:
summary = run_protocol_eve(S_start=10_000_000, S_end=100_000_000, q=100, mu1=20_000, mu2=20_160, seed=None)     # defaults: n=200000, q=250, num_trials=2000
                                   # run_protocol(n=200000, q=50, k=8, mu1=20000, mu2=50000,
                                   # S_start=10_000_000, S_end=100_000_000,
                                   # num_trials=2000, seed=None):
append_run_eve(summary)
summary

{'success_rate': 0.998,
 'Eve success rate': np.float64(0.999),
 'S_start': 10000000,
 'S_end': 100000000,
 'S_range': '90000000',
 'q': 100,
 'n': 200000,
 'mu1': 20000,
 'mu2': 20160,
 'num_trials': 2000,
 'acc_given_U': 1.0,
 'acc_given_V': 0.0,
 'Set U': array([ 7415.76543774,  6490.80383847,  1949.18798585, 20276.46585199,
         2459.13797301, 58336.25896035,  1449.2144034 ,  3644.45399935]),
 'Mean of U': np.float64(12752.661056270254),
 'seed': None}